# Agentic RAG — Taller de Tecnologías 2

Chatbot en lenguaje natural que responde sobre un dataset de correos de phishing y sobre documentos
largos, con un agente de LangGraph, tool calling y memoria a corto y largo plazo.

Este notebook importa los módulos de `src/` y muestra el sistema funcionando de punta a punta. Integra:

| Componente | Elección |
|---|---|
| LLM | `qwen/qwen3.8-27b` vía Groq |
| Embeddings | `intfloat/multilingual-e5-small`, locales |
| Base vectorial | ChromaDB persistente, colecciones `dataset` y `papers` |
| Agente | Grafo de LangGraph escrito a mano, sin `create_react_agent` ni `ToolNode` |
| Memoria | `SqliteSaver` (corto plazo) y `SqliteStore` (largo plazo) |

**Requisitos previos:** seguir los pasos de `README.md` para tener `.env`, `data/` y `chroma_db/`.

> **Estado: incompleto, sin outputs.** Las secciones 1 a 6 se ejecutaron con datos reales y las secciones 7 a 10 solo se probaron con un LLM simulado. Falta ejecutar el notebook completo con Groq y guardar sus outputs. Quitar este aviso después de esa ejecución.

## 1. Preparación

In [ ]:
import builtins
import sys
import tempfile
import time
import uuid
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter

sys.path.insert(0, str(Path.cwd()))

from src import agente as modulo_agente
from src import chatbot, config, datos, ingesta, memoria, recuperacion

# En False se omiten las celdas que consumen tokens de Groq.
EJECUTAR_LLM = True

# Memoria temporal: cada ejecución del notebook arranca limpia y no toca memoria.sqlite3.
config.ARCHIVO_MEMORIA = Path(tempfile.mkdtemp()) / "memoria_notebook.sqlite3"
pd.set_option("display.max_colwidth", 120)

print("Clave de Groq configurada:", bool(config.GROQ_API_KEY))
print("Papers disponibles:", len(list(config.DIR_PAPERS.glob("*.pdf"))))
print("Dataset consolidado:", config.CSV_CONSOLIDADO.exists())
print("Base vectorial:", config.DIR_CHROMA.exists())

## 2. Dataset

Se eligió el [Phishing Email Dataset](https://www.kaggle.com/datasets/naserabdullahalam/phishing-email-dataset).
Los datos están en inglés y el chatbot funciona en español e inglés. `src/datos.py` une seis corpus en un
único CSV y conserva el corpus de origen, el remitente, el asunto, el cuerpo y la etiqueta.

In [ ]:
correos = datos.cargar_consolidado()
print(f"Correos: {len(correos):,}")
correos[["fuente", "sender", "subject", "urls", "label"]].head()

In [ ]:
composicion = correos.groupby("fuente")["label"].agg(total="count", phishing="sum")
composicion["legitimos"] = composicion["total"] - composicion["phishing"]
composicion["remitente_disponible"] = correos.groupby("fuente")["sender"].apply(lambda serie: serie.notna().mean().round(2))
composicion.sort_values("total", ascending=False)

Enron y Ling no incluyen remitente ni URLs, por eso las estadísticas informan sobre cuántos correos se
calcularon. Además, la etiqueta `phishing` del dataset también incluye spam común.

Para la búsqueda semántica se indexa una muestra balanceada de 10.000 correos, por indicación docente.
Las estadísticas se calculan sobre el dataset completo.

## 3. Chunking y embeddings

Los textos se cortan con `RecursiveCharacterTextSplitter` (800 caracteres, 100 de solapamiento), que
prioriza respetar párrafos y oraciones. Cada fragmento conserva sus metadatos para citar y filtrar.

In [ ]:
separador = RecursiveCharacterTextSplitter(chunk_size=config.CHUNK_SIZE, chunk_overlap=config.CHUNK_OVERLAP)
correo_largo = correos.loc[correos["body"].astype(str).str.len().between(2000, 3000)].iloc[0]
fragmentos = separador.split_text(str(correo_largo["body"]))

print(f"Correo de {len(str(correo_largo['body']))} caracteres -> {len(fragmentos)} fragmentos")
print("Largo de cada fragmento:", [len(fragmento) for fragmento in fragmentos])

Los PDF se normalizan y se descartan los fragmentos con menos de 50% de letras. Esos fragmentos suelen
ser tablas numéricas, y en las pruebas aparecían como resultado de consultas sin relación.

In [ ]:
print(repr(ingesta.normalizar_texto("El Trans-\nformer usa   atención\nmulti-cabeza.")))
print("Texto común:", ingesta.es_fragmento_util("The Transformer relies entirely on attention mechanisms."))
print("Tabla numérica:", ingesta.es_fragmento_util("28.4 41.8 2.3 1.0 3.3 10 18 5.0 7.7 9.2 26.3"))

El modelo e5 es multilingüe: textos con el mismo significado quedan cerca aunque estén en otro idioma.
Esto permite preguntar en español sobre documentos en inglés. El modelo exige los prefijos `query:` y
`passage:`, que agrega la clase `EmbeddingsE5`.

In [ ]:
embeddings = ingesta.crear_embeddings()
consulta = np.array(embeddings.embed_query("correo que pide verificar la contraseña de la cuenta"))
textos = [
    "Your password has expired, click here to verify your account",
    "The quarterly meeting has been moved to Tuesday afternoon",
    "Add the tomatoes and let the sauce simmer for ten minutes",
]
vectores = np.array(embeddings.embed_documents(textos))
similitud = vectores @ consulta / (np.linalg.norm(vectores, axis=1) * np.linalg.norm(consulta))

print("Dimensión del vector:", consulta.shape[0])
pd.DataFrame({"texto en inglés": textos, "similitud coseno": similitud.round(3)})

## 4. Base vectorial y búsqueda semántica

Los índices se construyen una vez con `scripts/construir_indice.py` y `scripts/construir_indice_papers.py`,
y quedan persistidos en `chroma_db/`.

In [ ]:
for coleccion in (config.COLECCION_DATASET, config.COLECCION_PAPERS):
    print(f"{coleccion}: {ingesta.cargar_indice(coleccion)._collection.count():,} fragmentos")

In [ ]:
consultas = {
    "¿qué es el mecanismo de atención multi-cabeza?": "pertinente",
    "¿cómo escala la pérdida con el tamaño del modelo?": "pertinente",
    "replicación líder-seguidor en bases de datos": "pertinente",
    "mejores playas de Uruguay para el verano": "ajena",
    "¿cuál es la capital de Australia?": "ajena",
}
indice_papers = ingesta.cargar_indice(config.COLECCION_PAPERS)
filas = []
for consulta_texto, tipo in consultas.items():
    documento, puntaje = indice_papers.similarity_search_with_relevance_scores(consulta_texto, k=1)[0]
    filas.append({
        "consulta": consulta_texto,
        "tipo": tipo,
        "relevancia": round(puntaje, 3),
        "supera el umbral": puntaje >= config.UMBRAL_RELEVANCIA,
        "mejor resultado": f"{documento.metadata['titulo'][:40]}, pág. {documento.metadata['pagina']}",
    })
pd.DataFrame(filas)

Los puntajes de e5 están comprimidos y se solapan, por lo que un umbral no separa por sí solo lo pertinente
de lo ajeno. El umbral es un piso: si nada lo supera, la tool devuelve `SIN EVIDENCIA`. El resto del control
lo hace el prompt, que exige verificar que los fragmentos respondan la pregunta. Por ejemplo, "receta de lasaña
vegetariana" supera el umbral con 0,72 y recupera un fragmento sin relación, por lo que el umbral solo no alcanza.

## 5. Herramientas del agente

El agente recibe estas tools. Los ejemplos siguientes muestran lo que devuelven, sin usar el LLM.

In [ ]:
herramientas = recuperacion.HERRAMIENTAS + [memoria.guardar_recuerdo]
pd.DataFrame([
    {"tool": herramienta.name, "descripción": herramienta.description.split("\n")[0]}
    for herramienta in herramientas
])

In [ ]:
print(recuperacion.buscar_en_papers.invoke({"consulta": "multi-head attention", "titulo_paper": "attention", "cantidad": 1})[:500])
print("\n--- Consulta ajena ---")
print(recuperacion.buscar_en_papers.invoke({"consulta": "mejores playas de Uruguay para el verano"})[:300])
print("\n--- Correos ---")
print(recuperacion.buscar_en_dataset.invoke({"consulta": "account suspended verify password", "tipo_correo": "phishing"})[:500])

In [ ]:
print(recuperacion.estadisticas_dataset.invoke({}))
print()
print(recuperacion.ranking_dataset.invoke({"campo": "dominio", "tipo_correo": "phishing", "cantidad": 5}))

## 6. Agente con LangGraph

El grafo tiene dos nodos escritos a mano. `agente` llama al LLM con las tools disponibles y decide si
responde o si pide herramientas. `herramientas` las ejecuta y devuelve el control. No hay un router aparte:
la clasificación entre documentos, consulta general y small-talk la decide el LLM guiado por el prompt.

In [ ]:
agente = modulo_agente.construir_agente(memoria.crear_memoria_corto_plazo(), memoria.crear_memoria_largo_plazo())
print(agente.get_graph().draw_mermaid())

### Prompt de sistema

Técnicas aplicadas: rol, instrucciones por tipo de consulta, reglas contra alucinaciones, citas obligatorias,
ejemplos few-shot, idioma espejo y límite de longitud.

In [ ]:
print(modulo_agente.PROMPT_SISTEMA)

## 7. Con contexto y sin contexto

Esta prueba muestra el aporte del RAG con la misma pregunta. Primero el LLM responde sin documentos y
después con los fragmentos recuperados.

In [ ]:
pregunta = "¿Qué condición deben cumplir el modelo maestro y el modelo estudiante para que ocurra el subliminal learning?"

if EJECUTAR_LLM:
    llm = modulo_agente.crear_llm()
    sin_contexto = modulo_agente.invocar_llm(llm, [HumanMessage(pregunta + " Respondé en dos oraciones.")])
    contexto = recuperacion.buscar_en_papers.invoke({"consulta": pregunta, "titulo_paper": "subliminal"})
    con_contexto = modulo_agente.invocar_llm(llm, [
        SystemMessage("Respondé en dos oraciones usando solo este contexto y citá la página:\n\n" + contexto),
        HumanMessage(pregunta),
    ])
    display(Markdown(f"**Sin contexto:** {sin_contexto.content}"))
    display(Markdown(f"**Con contexto:** {con_contexto.content}"))
else:
    print("Celda omitida: EJECUTAR_LLM = False")

## 8. Pruebas del chatbot

Cada caso declara qué tools se esperan. La columna `cumple` compara ese valor con lo que decidió el agente.
La última pregunta sigue a la anterior en la misma conversación.

In [ ]:
def preguntar(texto, thread_id, user_id="demo"):
    """Envía un mensaje al agente y devuelve la respuesta, las tools del turno y el tiempo."""
    inicio = time.time()
    sesion = {"configurable": {"thread_id": thread_id, "user_id": user_id}}
    mensajes = agente.invoke({"messages": [("user", texto)]}, sesion)["messages"]
    return mensajes[-1].content, chatbot._herramientas_usadas(mensajes), round(time.time() - inicio)


casos = [
    ("small-talk", "Hola! cómo andás?", set(), "c1"),
    ("general", "¿Cuál es la capital de Francia?", set(), "c2"),
    ("papers ES", "¿Cuántos parámetros tiene GPT-3?", {"buscar_en_papers"}, "c3"),
    ("papers EN", "What does the Scaling Laws paper say about model size and loss?", {"buscar_en_papers"}, "c4"),
    ("resumen", 'Resumir el artículo "Attention Is All You Need" en tres párrafos.', {"buscar_en_papers"}, "c5"),
    ("sin evidencia", "¿Qué dice el paper de Attention sobre el mundial de fútbol 2022?", {"buscar_en_papers"}, "c6"),
    ("estadística", "¿Cuántos correos de phishing hay en el dataset?", {"estadisticas_dataset"}, "c7"),
    ("ranking", "¿Qué dominios usan más los correos de phishing?", {"ranking_dataset"}, "c8"),
    ("dataset", "Mostrame un ejemplo de correo de phishing bancario", {"buscar_en_dataset"}, "c9"),
    ("análisis", "Este correo dice 'Your account is suspended, click here'. ¿Es phishing?", {"buscar_en_dataset"}, "c10"),
    ("contexto", "¿Qué es multi-head attention?", {"buscar_en_papers"}, "c11"),
    ("seguimiento", "¿Y cuántas cabezas usa el modelo big?", {"buscar_en_papers"}, "c11"),
]

resultados = []
if EJECUTAR_LLM:
    for tipo, texto, esperadas, thread_id in casos:
        try:
            respuesta, usadas, segundos = preguntar(texto, thread_id)
            error = ""
        except Exception as excepcion:
            respuesta, usadas, segundos, error = "", [], 0, type(excepcion).__name__
        resultados.append({"tipo": tipo, "pregunta": texto, "tools": sorted(set(usadas)), "cumple": set(usadas) == esperadas,
                           "segundos": segundos, "error": error, "respuesta": respuesta})
        display(Markdown(f"**[{tipo}] {texto}**\n\n*Tools: {sorted(set(usadas)) or 'ninguna'}*\n\n{respuesta or error}\n\n---"))
else:
    print("Celda omitida: EJECUTAR_LLM = False")

In [ ]:
tabla_resultados = pd.DataFrame(resultados)
if EJECUTAR_LLM:
    print(f"Decisión de tools correcta: {tabla_resultados['cumple'].sum()} de {len(tabla_resultados)}")
    print(f"Tiempo medio por pregunta: {tabla_resultados['segundos'].mean():.0f} s")
    display(tabla_resultados[["tipo", "tools", "cumple", "segundos", "error"]])

## 9. Memoria y sesiones

- **Corto plazo:** el checkpointer guarda cada conversación por `thread_id`. Cada llamada al LLM recibe una
  ventana de los últimos mensajes y el turno actual completo.
- **Largo plazo:** el agente decide qué guardar con `guardar_recuerdo`. Los recuerdos se guardan por `user_id`
  y se agregan al prompt en cada turno.
- **Sesiones:** cada invocación lleva su `thread_id` y su `user_id`.

In [ ]:
def decir(usuario, conversacion, texto):
    respuesta, usadas, _ = preguntar(texto, f"{usuario}-{conversacion}", usuario)
    display(Markdown(f"**{usuario} / conversación {conversacion}:** {texto}\n\n{respuesta}\n\n*Tools: {usadas or 'ninguna'}*"))


if EJECUTAR_LLM:
    decir("ana", "1", "Hola! Soy Ana y trabajo en seguridad informática en un banco.")
    decir("ana", "1", "En una frase: ¿qué es el few-shot learning?")
    print("Recuerdos guardados de Ana:", memoria.leer_recuerdos(agente.store, "ana"))
    decir("ana", "2", "¿Te acordás cómo me llamo y a qué me dedico?")
    decir("ana", "2", "¿Qué te pregunté en la otra conversación?")
else:
    print("Celda omitida: EJECUTAR_LLM = False")

### Pausar y retomar después de reiniciar

Se crea un agente nuevo, con conexiones nuevas a SQLite y sin nada en memoria RAM, y Ana vuelve a la
conversación 1.

In [ ]:
if EJECUTAR_LLM:
    agente = modulo_agente.construir_agente(memoria.crear_memoria_corto_plazo(), memoria.crear_memoria_largo_plazo())
    decir("ana", "1", "¿Qué te había preguntado recién?")
else:
    print("Celda omitida: EJECUTAR_LLM = False")

### Sesiones de usuario en paralelo

In [ ]:
if EJECUTAR_LLM:
    with ThreadPoolExecutor() as hilos:
        tareas = {
            usuario: hilos.submit(preguntar, "Hola, ¿sabés cómo me llamo?", f"{usuario}-paralelo", usuario)
            for usuario in ("ana", "beto")
        }
    for usuario, tarea in tareas.items():
        print(f"{usuario}: {tarea.result()[0]}")
    print("Recuerdos de Beto:", memoria.leer_recuerdos(agente.store, "beto"))
else:
    print("Celda omitida: EJECUTAR_LLM = False")

## 10. Chatbot

`main.py` ejecuta este mismo chatbot por consola. La celda siguiente lo usa con mensajes escritos de antemano,
para que el notebook se pueda ejecutar sin intervención. Para usarlo de forma interactiva, ejecutar
`chatbot.iniciar_chat()`.

In [ ]:
def chat_guionado(entradas):
    """Ejecuta el chatbot real reemplazando input() por una lista de mensajes."""
    pendientes = iter(entradas)

    def responder(indicador=""):
        texto = next(pendientes)
        print(f"{indicador}{texto}")
        return texto

    original, builtins.input = builtins.input, responder
    try:
        chatbot.iniciar_chat()
    finally:
        builtins.input = original


if EJECUTAR_LLM:
    usuario = f"demo-{uuid.uuid4().hex[:6]}"
    chat_guionado([f"/usuario {usuario}", "Hola, me llamo Sofía", "/nueva", "Gracias, ¿qué podés hacer?", "/conversaciones", "/salir"])
    conversaciones = memoria.listar_conversaciones(memoria.crear_memoria_largo_plazo(), usuario)
    primera = next(iter(conversaciones))
    chat_guionado([f"/usuario {usuario}", f"/retomar {primera}", "¿Cómo me llamo?", "/salir"])
else:
    print("Celda omitida: EJECUTAR_LLM = False")

## 11. Limitaciones

- **Cuota de Groq:** el plan gratuito permite 8.000 tokens por minuto y 200.000 por día. Una pregunta con
  búsqueda hace al menos dos llamadas y puede tardar decenas de segundos.
- **Largo de las respuestas:** están limitadas a 900 tokens de salida.
- **Resúmenes:** se generan con hasta 10 fragmentos recuperados, no con el documento completo.
- **Páginas:** las citas usan el número de página del PDF, que puede diferir del número impreso.
- **Control de alucinaciones:** se apoya en el prompt, el umbral y las citas, por lo que reduce el riesgo pero
  no lo elimina.